# Answer Extraction Check

In [4]:
import json
from pathlib import Path

from evaluator import AnswerExtractor

In [6]:
results_path = Path("/home/dario/denv/experiments/results/aime/kosbu_Llama-3.3-70B-Instruct-AWQ/reasoning_false_personas_false_few-shot.json")
with results_path.open() as f:
    results = json.load(f)

print(f"Loaded runs: {len(results)} from {results_path}")

Loaded runs: 10 from /home/dario/denv/experiments/results/aime/kosbu_Llama-3.3-70B-Instruct-AWQ/reasoning_false_personas_false_few-shot.json


In [8]:
extractor = AnswerExtractor()

rows = []

total = 0
correct = 0
nd = 0
nd_length = 0

def is_correct(extracted, golden):
    if extracted in ("N.D.", "N.D. LENGTH"):
        return False
    try:
        return int(extracted) == int(golden)
    except (ValueError, TypeError):
        return str(extracted).strip() == str(golden).strip()

for run_idx, run_results in enumerate(results, start=1):
    for item in run_results:
        output = item.get("output", {})
        output_text = output.get("output_text") or output.get("text", "")
        finish_reason = output.get("finish_reason")
        choices = item.get("options") or item.get("choices")
        golden = item.get("answer")
        extracted = extractor.extract(
            output_text=output_text,
            golden_answer=golden,
            choices=choices,
            dataset="aime",
            finish_reason=finish_reason,
        )

        total += 1
        if extracted == "N.D.":
            nd += 1
        elif extracted == "N.D. LENGTH":
            nd_length += 1
        elif is_correct(extracted, golden):
            correct += 1

        index = item.get("index", "?")
        rows.append(f"run={run_idx:02d} index={index} extracted={extracted} golden={golden}")

output_path = Path("/home/dario/denv/experiments/results/aime/kosbu_Llama-3.3-70B-Instruct-AWQ/answer_extraction_check.txt")
output_path.write_text("\n".join(rows) + "\n")

correct_pct = (correct / total * 100) if total else 0.0
nd_pct = (nd / total * 100) if total else 0.0
nd_length_pct = (nd_length / total * 100) if total else 0.0

print(f"Wrote {len(rows)} rows to {output_path}")
print(f"Total: {total}")
print(f"Correct: {correct} ({correct_pct:.2f}%)")
print(f"N.D.: {nd} ({nd_pct:.2f}%)")
print(f"N.D. LENGTH: {nd_length} ({nd_length_pct:.2f}%)")

Wrote 300 rows to /home/dario/denv/experiments/results/aime/kosbu_Llama-3.3-70B-Instruct-AWQ/answer_extraction_check.txt
Total: 300
Correct: 20 (6.67%)
N.D.: 13 (4.33%)
N.D. LENGTH: 0 (0.00%)
